# Module 2 Lab — Data Cleaning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/applied-data-science-python/blob/main/module-02-data-cleaning-and-preprocessing/notebooks/Module02_Data_Cleaning_Lab.ipynb)

**Applied Data Science with Python** · Open Educational Resource · CC BY 4.0

Each section follows the same path: **Learning Objectives → Concept → Guided Practice → Your Turn**. Run the cells in order. In *Your Turn* cells, replace every `___` (or the `# YOUR CODE HERE` line) and run the cell; the `assert` checks tell you when you are right.

Estimated time: 6 hours of practical work.

In [ ]:
# Run this cell first. It loads course datasets from the repository
# (works locally, in JupyterLab, and in Google Colab).
import os
import pandas as pd

REPO_RAW = "https://raw.githubusercontent.com/YOUR-GITHUB-USERNAME/applied-data-science-python/main/datasets/"   # <- replace YOUR-GITHUB-USERNAME once you publish the repo

def load_data(name, **kwargs):
    """Load a course dataset by file name, e.g. load_data("titanic.csv")."""
    for folder in ("../../datasets", "../datasets", "datasets", "."):
        path = os.path.join(folder, name)
        if os.path.exists(path):
            return pd.read_csv(path, **kwargs)
    return pd.read_csv(REPO_RAW + name, **kwargs)

## Part 1 — Inspect and fix data types

### Learning Objectives
- Profile a raw dataset
- Convert dates and categories to the right dtypes

### Concept
Before any cleaning, look: how many rows and columns, which dtypes, how many missing values, and what a single row represents. Text dates must be parsed before you can sort or group by time.

### Guided Practice

In [ ]:
import numpy as np
raw = load_data("retail_sales_raw.csv")
print(raw.shape)
raw.info()

In [ ]:
# What does missingness look like?
missing = raw.isna().mean().mul(100).round(1).sort_values(ascending=False)
missing[missing > 0]

### Your Turn
**Exercise 1.1** — Parse `raw['date']` into datetimes (some dates are day/month/year) and convert `customer_type` and `payment_method` to the `category` dtype.

In [ ]:
raw["date"] = ___
raw["customer_type"] = ___
raw["payment_method"] = ___

assert str(raw["date"].dtype).startswith("datetime64")
assert raw["date"].isna().sum() == 0
assert str(raw["customer_type"].dtype) == "category"
print("Correct!")

## Part 2 — Missing data

### Learning Objectives
- Choose between dropping and filling
- Fill with statistics, and with information from other columns

### Concept
`dropna()` is quick but can throw away a lot of data. Filling keeps rows but adds assumptions. Choose the fill to match the column: median for skewed numbers, mode for categories, interpolation for ordered series.

### Guided Practice

In [ ]:
print("Rows if we drop every incomplete row:", len(raw.dropna()), "of", len(raw))
# Each store belongs to exactly one region, so we can recover missing regions from store_id
store_region = raw.dropna(subset=["region"]).assign(region=lambda d: d["region"].str.title()) \
                  .groupby("store_id")["region"].agg(lambda s: s.mode()[0])
store_region

### Your Turn
**Exercise 2.1** — Standardise `region` to title case, then fill its missing values using `store_region` (hint: `raw['store_id'].map(store_region)`).

In [ ]:
raw["region"] = ___
raw["region"] = ___

assert raw["region"].isna().sum() == 0
assert set(raw["region"].unique()) == {"North", "South", "East", "West"}
print("Correct!")

**Exercise 2.2** — Fill `payment_method` with its mode and `rating` with its median.

In [ ]:
# YOUR CODE HERE

assert raw["payment_method"].isna().sum() == 0 and raw["rating"].isna().sum() == 0
print("Correct!")

**Exercise 2.3** — Daily temperatures are missing for two days. Fill `temps` three ways (forward fill, backward fill, linear interpolation) and store them in `f`, `b` and `lin`.

In [ ]:
temps = pd.Series([21.0, np.nan, np.nan, 27.0, 26.0])
f = ___
b = ___
lin = ___

assert f.tolist() == [21, 21, 21, 27, 26] and b.tolist() == [21, 27, 27, 27, 26]
assert lin.tolist() == [21, 23, 25, 27, 26]
print("Correct!")

## Part 3 — Transformation: duplicates, values, names, outliers

### Learning Objectives
- Remove duplicate records
- Replace invalid values, rename columns
- Detect outliers with Z-scores and IQR

### Concept
Fix the *meaning* of values before computing statistics: a stray space makes `' Electronics '` a different category, and an age of 999 inflates the mean.

### Guided Practice

In [ ]:
print("Duplicate rows:", raw.duplicated().sum())
print(raw["category"].value_counts())
print("Impossible ages:", (raw["customer_age"] > 110).sum())

### Your Turn
**Exercise 3.1** — Drop duplicate rows, strip spaces from `category` (hint: `.str.strip()`), replace impossible ages (> 110) with NaN using `where()`, then fill `customer_age` and `unit_price` with their column means. Finally rename `units` to `qty`.

In [ ]:
# YOUR CODE HERE

assert raw.duplicated().sum() == 0
assert raw["category"].nunique() == 5
assert raw["customer_age"].max() <= 110 and raw["customer_age"].isna().sum() == 0
assert raw["unit_price"].isna().sum() == 0 and "qty" in raw.columns
print("Correct!")

**Exercise 3.2** — Compute the IQR fences for `qty` (`lo`, `hi`) and the Z-scores (`z`). Count how many rows each method flags (`n_iqr`, `n_z` with |z| > 3). Then keep only rows with |z| ≤ 3 in `clean`.

In [ ]:
q1, q3 = ___
lo, hi = ___
z = ___
n_iqr = ___
n_z = ___
clean = ___

assert lo < hi and n_iqr >= n_z > 0
assert clean["qty"].max() < 30
print(f"IQR flags {n_iqr}, Z-score flags {n_z}. Correct!")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
raw.boxplot(column="qty", ax=ax[0]); ax[0].set_title("Before")
clean.boxplot(column="qty", ax=ax[1]); ax[1].set_title("After |z| <= 3")
plt.show()

## Part 4 — Feature engineering and encoding

### Learning Objectives
- Build new features
- Encode categories for modelling

### Concept
Features can be arithmetic (`price * qty`), conditional (`np.where`) or functional (`apply`). Models need numbers: one-hot encode unordered categories, ordinal-encode ordered ones.

### Your Turn
**Exercise 4.1** — On `clean`, recompute `revenue` = unit_price × qty × (1 − discount), add `is_weekend` (1 for Saturday/Sunday), and add `month` from the date.

In [ ]:
clean = clean.copy()
# YOUR CODE HERE

assert np.allclose(clean["revenue"], clean["unit_price"] * clean["qty"] * (1 - clean["discount"]))
assert set(clean["is_weekend"].unique()) <= {0, 1} and clean["month"].between(1, 12).all()
print("Correct!")

**Exercise 4.2** — One-hot encode `category` and `payment_method` into `encoded` (integer 0/1 columns, prefix with the column name), and ordinal-encode `rating_level`: ratings 1–2 = 0 (low), 3 = 1 (neutral), 4–5 = 2 (high).

In [ ]:
encoded = ___
clean["rating_level"] = ___

assert "category_Books" in encoded.columns and "payment_method_Cash" in encoded.columns
assert set(clean["rating_level"].unique()) <= {0, 1, 2}
print("Correct!", encoded.shape)

## Mini-project — Clean the Titanic data

Load `titanic.csv` and produce a model-ready DataFrame `titanic_ready`:
1. Fill `age` with the median age **per passenger class** (hint: `groupby('pclass')['age'].transform('median')`).
2. Fill `embarked` with its mode; drop the mostly-empty `deck` column.
3. Create `family_size` and `is_alone`.
4. One-hot encode `sex` and `embarked`.

Write two sentences explaining your choices. Then take the **Module 2 quiz**.

In [ ]:
titanic = load_data("titanic.csv")
# YOUR CODE HERE
titanic_ready = ___